# 08 — Baseline Thai Lip Reading Training

Baseline แรกสำหรับทดสอบ end-to-end pipeline:

```text
Mouth Frames 96×96
    ↓
Frame CNN Encoder
    ↓
BiGRU Temporal Encoder
    ↓
Linear Classifier
    ↓
CTC Loss
    ↓
Thai Character Sequence
```

Input:
- `dataset/splits/video001/train.csv`
- `dataset/splits/video001/val.csv`
- `dataset/splits/video001/test.csv`

Output:
- `models/lipreading_baseline/best_model.pt`
- `models/lipreading_baseline/last_model.pt`
- `models/lipreading_baseline/charset.json`
- `models/lipreading_baseline/training_history.csv`

## เป้าหมายของรอบแรก

รอบนี้ **ไม่ได้หวังความแม่นสูงทันที** เพราะข้อมูลยังมีเพียง ~33 นาที / 2 speakers

สิ่งที่ต้องการพิสูจน์:
1. Dataset loader อ่าน mouth sequence ได้จริง
2. CTC training ทำงาน
3. Train loss ลด
4. Validation loss ลด
5. Decode อักษรไทยได้
6. คำนวณ CER ได้
7. Model สามารถ overfit subset เล็ก ๆ ได้

## Cell 1 — Environment Check

In [ ]:
import sys
import torch

print("=" * 70)
print("ENVIRONMENT")
print("=" * 70)
print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("Torch  :", torch.__version__)
print("CUDA   :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Baseline training นี้ควรใช้ CUDA")

DEVICE = torch.device("cuda")

print("GPU    :", torch.cuda.get_device_name(0))
print("VRAM   :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

## Cell 2 — Path / Config

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

SPLIT_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "splits"
    / VIDEO_ID
)

TRAIN_CSV = SPLIT_ROOT / "train.csv"
VAL_CSV = SPLIT_ROOT / "val.csv"
TEST_CSV = SPLIT_ROOT / "test.csv"

MODEL_ROOT = (
    PROJECT_ROOT
    / "models"
    / "lipreading_baseline"
)

MODEL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(p, "->", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)

print("MODEL_ROOT:", MODEL_ROOT)

## Cell 3 — Training Config

สำหรับ RTX 4050 Laptop เริ่มแบบ conservative ก่อน

ถ้า CUDA out of memory:
- ลด `BATCH_SIZE = 2` → `1`
- ลด `MAX_FRAMES_PER_CLIP`
- ลด `GRU_HIDDEN`

In [ ]:
SEED = 42

IMAGE_SIZE = 96
BATCH_SIZE = 2
NUM_WORKERS = 0  # Windows + Notebook ใช้ 0 ปลอดภัยสุด

GRU_HIDDEN = 192
GRU_LAYERS = 2
DROPOUT = 0.20

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

EPOCHS = 30

# จำกัด clip ยาวมากเกินไปเพื่อคุม VRAM
MAX_FRAMES_PER_CLIP = 180

# gradient accumulation ช่วยให้ effective batch ใหญ่ขึ้น
GRAD_ACCUM_STEPS = 4

# gradient clipping
GRAD_CLIP_NORM = 5.0

# early stopping
PATIENCE = 7

print("BATCH_SIZE          :", BATCH_SIZE)
print("EPOCHS              :", EPOCHS)
print("MAX_FRAMES_PER_CLIP :", MAX_FRAMES_PER_CLIP)
print("GRAD_ACCUM_STEPS    :", GRAD_ACCUM_STEPS)

## Cell 4 — Seed

In [ ]:
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

print("Seed ready")

## Cell 5 — Load Splits

In [ ]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))

display(train_df.head())

## Cell 6 — Character Tokenizer

CTC vocabulary:
- index 0 = `<blank>`
- index 1 = `<unk>`
- index 2+ = characters ที่พบใน Train

เก็บ space เป็น character ด้วย

In [ ]:
import json

BLANK_TOKEN = "<blank>"
UNK_TOKEN = "<unk>"

train_texts = (
    train_df["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

charset = sorted(
    set(
        "".join(train_texts)
    )
)

# remove accidental control chars
charset = [
    ch for ch in charset
    if ch not in ["\n", "\r", "\t"]
]

itos = [
    BLANK_TOKEN,
    UNK_TOKEN,
] + charset

stoi = {
    token: idx
    for idx, token in enumerate(itos)
}

BLANK_ID = stoi[BLANK_TOKEN]
UNK_ID = stoi[UNK_TOKEN]

VOCAB_SIZE = len(itos)

print("Vocab size:", VOCAB_SIZE)
print("Blank ID  :", BLANK_ID)
print("UNK ID    :", UNK_ID)
print("Charset sample:", charset[:100])

CHARSET_PATH = MODEL_ROOT / "charset.json"

with open(
    CHARSET_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "itos": itos,
            "blank_id": BLANK_ID,
            "unk_id": UNK_ID,
        },
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", CHARSET_PATH)

## Cell 7 — Encode / Decode Functions

In [ ]:
def encode_text(text):
    text = str(text)

    return [
        stoi.get(
            ch,
            UNK_ID
        )
        for ch in text
    ]


def decode_target(ids):
    chars = []

    for idx in ids:
        idx = int(idx)

        if idx == BLANK_ID:
            continue

        token = itos[idx]

        if token == UNK_TOKEN:
            chars.append("�")
        else:
            chars.append(token)

    return "".join(chars)


def ctc_greedy_decode(logits, lengths):
    # logits: [B, T, C]
    pred_ids = logits.argmax(dim=-1)

    results = []

    for b in range(pred_ids.size(0)):
        seq = pred_ids[
            b,
            :int(lengths[b])
        ].tolist()

        collapsed = []
        prev = None

        for idx in seq:
            if idx != prev:
                if idx != BLANK_ID:
                    collapsed.append(idx)

            prev = idx

        results.append(
            decode_target(
                collapsed
            )
        )

    return results


print("Tokenizer helpers ready")

## Cell 8 — Dataset

อ่าน `frame_paths` จาก manifest โดยไม่ copy รูป

Preprocessing:
- grayscale
- 96×96
- normalize 0..1
- shape `[T, 1, 96, 96]`

ถ้า clip ยาวเกิน `MAX_FRAMES_PER_CLIP` จะ sample แบบ uniform เพื่อลด VRAM

In [ ]:
import cv2
from torch.utils.data import Dataset

class LipReadingDataset(Dataset):

    def __init__(
        self,
        dataframe,
        project_root,
        max_frames=None,
    ):
        self.df = dataframe.reset_index(drop=True).copy()
        self.project_root = Path(project_root)
        self.max_frames = max_frames

    def __len__(self):
        return len(self.df)

    def _resolve_path(self, p):
        path = Path(str(p))

        if path.is_absolute():
            return path

        return self.project_root / path

    def _select_paths(self, paths):
        if (
            self.max_frames is None
            or len(paths) <= self.max_frames
        ):
            return paths

        indices = np.linspace(
            0,
            len(paths) - 1,
            self.max_frames
        ).astype(int)

        return [
            paths[i]
            for i in indices
        ]

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        paths = str(
            row["frame_paths"]
        ).split("|")

        paths = [
            p for p in paths
            if p.strip()
        ]

        paths = self._select_paths(
            paths
        )

        frames = []

        for p in paths:

            full_path = self._resolve_path(
                p
            )

            img = cv2.imread(
                str(full_path),
                cv2.IMREAD_GRAYSCALE
            )

            if img is None:
                continue

            if img.shape != (
                IMAGE_SIZE,
                IMAGE_SIZE
            ):
                img = cv2.resize(
                    img,
                    (
                        IMAGE_SIZE,
                        IMAGE_SIZE
                    ),
                    interpolation=cv2.INTER_AREA
                )

            img = (
                img.astype(
                    np.float32
                )
                / 255.0
            )

            frames.append(
                img
            )

        if not frames:
            raise RuntimeError(
                f"No readable frames for {row['clip_id']}"
            )

        video = np.stack(
            frames,
            axis=0
        )

        video = torch.from_numpy(
            video
        ).unsqueeze(1)

        text = str(
            row["text"]
        )

        target = torch.tensor(
            encode_text(text),
            dtype=torch.long
        )

        return {
            "video": video,
            "target": target,
            "text": text,
            "clip_id": row["clip_id"],
            "speaker_id": int(row["speaker_id"]),
        }


print("Dataset class ready")

## Cell 9 — CTC Collate Function

Padding:
- video → pad ตาม time dimension
- target → concatenate ตามรูปแบบที่ `CTCLoss` ต้องการ

In [ ]:
def collate_fn(batch):

    videos = [
        item["video"]
        for item in batch
    ]

    targets = [
        item["target"]
        for item in batch
    ]

    video_lengths = torch.tensor(
        [
            v.shape[0]
            for v in videos
        ],
        dtype=torch.long
    )

    target_lengths = torch.tensor(
        [
            t.numel()
            for t in targets
        ],
        dtype=torch.long
    )

    max_t = int(
        video_lengths.max()
    )

    batch_size = len(videos)

    padded = torch.zeros(
        (
            batch_size,
            max_t,
            1,
            IMAGE_SIZE,
            IMAGE_SIZE
        ),
        dtype=torch.float32
    )

    for i, video in enumerate(videos):
        padded[
            i,
            :video.shape[0]
        ] = video

    targets_concat = torch.cat(
        targets,
        dim=0
    )

    return {
        "video": padded,
        "video_lengths": video_lengths,
        "targets": targets_concat,
        "target_lengths": target_lengths,
        "texts": [
            item["text"]
            for item in batch
        ],
        "clip_ids": [
            item["clip_id"]
            for item in batch
        ],
        "speaker_ids": torch.tensor(
            [
                item["speaker_id"]
                for item in batch
            ],
            dtype=torch.long
        )
    }


print("Collate ready")

In [ ]:
train_loader = DataLoader(...)
val_loader = DataLoader(...)
test_loader = DataLoader(...)

## Cell 10 — DataLoaders

In [ ]:
from torch.utils.data import DataLoader

train_dataset = LipReadingDataset(
    train_df,
    PROJECT_ROOT,
    max_frames=MAX_FRAMES_PER_CLIP
)

val_dataset = LipReadingDataset(
    val_df,
    PROJECT_ROOT,
    max_frames=MAX_FRAMES_PER_CLIP
)

test_dataset = LipReadingDataset(
    test_df,
    PROJECT_ROOT,
    max_frames=MAX_FRAMES_PER_CLIP
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn,
)

batch = next(
    iter(train_loader)
)

print("Video batch :", batch["video"].shape)
print("Video lens  :", batch["video_lengths"])
print("Targets     :", batch["targets"].shape)
print("Target lens :", batch["target_lengths"])
print("Texts       :", batch["texts"])

## Cell 11 — CTC Compatibility Check

CTC ต้องการ `target_length <= input_length`

เนื่องจาก model baseline นี้ **ไม่ลด time dimension** input length จะเท่ากับจำนวน mouth frames หลัง sampling

In [ ]:
def check_ctc_compatibility(dataframe):
    bad = []

    for _, row in dataframe.iterrows():

        num_frames = min(
            int(row["num_frames"]),
            MAX_FRAMES_PER_CLIP
        )

        target_len = len(
            encode_text(
                str(row["text"])
            )
        )

        if target_len > num_frames:
            bad.append({
                "clip_id": row["clip_id"],
                "frames": num_frames,
                "target_len": target_len,
                "text": row["text"],
            })

    return pd.DataFrame(bad)


bad_train = check_ctc_compatibility(train_df)

print(
    "CTC-incompatible train clips:",
    len(bad_train)
)

if len(bad_train) > 0:
    display(
        bad_train.head(30)
    )
else:
    print("CTC compatibility: PASS")

## Cell 12 — Baseline Model

Frame encoder:
- Conv2D → BN → ReLU → Pool
- Conv2D → BN → ReLU → Pool
- Conv2D → BN → ReLU → AdaptiveAvgPool
- output 128-D ต่อ frame

Temporal encoder:
- 2-layer BiGRU

Output:
- Character logits สำหรับ CTC

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class FrameEncoder(nn.Module):

    def __init__(self, out_dim=128):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=5,
                stride=1,
                padding=2
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(
                (1, 1)
            )
        )

        self.proj = nn.Linear(
            128,
            out_dim
        )

    def forward(self, x):

        # x: [N,1,96,96]

        x = self.net(x)

        x = x.flatten(1)

        x = self.proj(x)

        return x


class LipReadingBaseline(nn.Module):

    def __init__(
        self,
        vocab_size,
        feature_dim=128,
        hidden_dim=192,
        gru_layers=2,
        dropout=0.2,
    ):
        super().__init__()

        self.frame_encoder = FrameEncoder(
            out_dim=feature_dim
        )

        self.temporal = nn.GRU(
            input_size=feature_dim,
            hidden_size=hidden_dim,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=(
                dropout
                if gru_layers > 1
                else 0.0
            )
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.classifier = nn.Linear(
            hidden_dim * 2,
            vocab_size
        )

    def forward(
        self,
        video,
        lengths
    ):

        # video: [B,T,1,H,W]

        B, T, C, H, W = (
            video.shape
        )

        x = video.reshape(
            B * T,
            C,
            H,
            W
        )

        features = self.frame_encoder(
            x
        )

        features = features.reshape(
            B,
            T,
            -1
        )

        packed = nn.utils.rnn.pack_padded_sequence(
            features,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.temporal(
            packed
        )

        temporal_out, _ = (
            nn.utils.rnn.pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=T
            )
        )

        temporal_out = self.dropout(
            temporal_out
        )

        logits = self.classifier(
            temporal_out
        )

        return logits


model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(model)
print()
print("Total parameters    :", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

## Cell 13 — Forward Pass Test

In [ ]:
model.eval()

with torch.no_grad():

    video = batch["video"].to(
        DEVICE
    )

    lengths = batch[
        "video_lengths"
    ].to(DEVICE)

    logits = model(
        video,
        lengths
    )

print("Input :", video.shape)
print("Output:", logits.shape)

assert (
    logits.shape[0]
    == video.shape[0]
)

assert (
    logits.shape[1]
    == video.shape[1]
)

assert (
    logits.shape[2]
    == VOCAB_SIZE
)

print("Forward pass: PASS")

## Cell 14 — Loss / Optimizer / AMP

In [ ]:
ctc_loss_fn = nn.CTCLoss(
    blank=BLANK_ID,
    reduction="mean",
    zero_infinity=True,
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

scaler = torch.amp.GradScaler(
    "cuda"
)

print("Training components ready")

## Cell 15 — CER Function

In [ ]:
def levenshtein_distance(a, b):

    prev = list(
        range(
            len(b) + 1
        )
    )

    for i, ca in enumerate(
        a,
        start=1
    ):
        curr = [i]

        for j, cb in enumerate(
            b,
            start=1
        ):
            insert_cost = (
                curr[j - 1] + 1
            )

            delete_cost = (
                prev[j] + 1
            )

            sub_cost = (
                prev[j - 1]
                + (
                    0
                    if ca == cb
                    else 1
                )
            )

            curr.append(
                min(
                    insert_cost,
                    delete_cost,
                    sub_cost
                )
            )

        prev = curr

    return prev[-1]


def character_error_rate(
    predictions,
    references
):

    total_edits = 0
    total_chars = 0

    for pred, ref in zip(
        predictions,
        references
    ):

        total_edits += (
            levenshtein_distance(
                pred,
                ref
            )
        )

        total_chars += max(
            1,
            len(ref)
        )

    return (
        total_edits
        / total_chars
    )


print("CER ready")

## Cell 16 — Train / Validation Functions

In [ ]:
from tqdm.auto import tqdm

def compute_ctc_loss(
    logits,
    video_lengths,
    targets,
    target_lengths,
):

    log_probs = F.log_softmax(
        logits,
        dim=-1
    )

    # CTCLoss expects [T,B,C]
    log_probs = log_probs.transpose(
        0,
        1
    )

    return ctc_loss_fn(
        log_probs,
        targets,
        video_lengths,
        target_lengths
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
):

    model.train()

    running_loss = 0.0
    steps = 0

    optimizer.zero_grad(
        set_to_none=True
    )

    pbar = tqdm(
        loader,
        desc="train",
        leave=False
    )

    for step, batch in enumerate(
        pbar,
        start=1
    ):

        video = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        video_lengths = batch[
            "video_lengths"
        ].to(
            DEVICE
        )

        targets = batch[
            "targets"
        ].to(
            DEVICE
        )

        target_lengths = batch[
            "target_lengths"
        ].to(
            DEVICE
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(
                video,
                video_lengths
            )

            loss = compute_ctc_loss(
                logits,
                video_lengths,
                targets,
                target_lengths
            )

            loss = (
                loss
                / GRAD_ACCUM_STEPS
            )

        scaler.scale(
            loss
        ).backward()

        if (
            step % GRAD_ACCUM_STEPS
            == 0
            or step == len(loader)
        ):

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM
            )

            scaler.step(
                optimizer
            )

            scaler.update()

            optimizer.zero_grad(
                set_to_none=True
            )

        true_loss = (
            float(loss.item())
            * GRAD_ACCUM_STEPS
        )

        running_loss += true_loss
        steps += 1

        pbar.set_postfix(
            loss=f"{running_loss / steps:.4f}"
        )

    return (
        running_loss
        / max(
            1,
            steps
        )
    )


@torch.no_grad()
def evaluate(
    model,
    loader,
    max_decode_batches=None
):

    model.eval()

    running_loss = 0.0
    steps = 0

    predictions = []
    references = []

    for batch_idx, batch in enumerate(
        tqdm(
            loader,
            desc="eval",
            leave=False
        )
    ):

        video = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        video_lengths = batch[
            "video_lengths"
        ].to(
            DEVICE
        )

        targets = batch[
            "targets"
        ].to(
            DEVICE
        )

        target_lengths = batch[
            "target_lengths"
        ].to(
            DEVICE
        )

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = model(
                video,
                video_lengths
            )

            loss = compute_ctc_loss(
                logits,
                video_lengths,
                targets,
                target_lengths
            )

        running_loss += float(
            loss.item()
        )

        steps += 1

        if (
            max_decode_batches is None
            or batch_idx
            < max_decode_batches
        ):

            decoded = ctc_greedy_decode(
                logits.float().cpu(),
                batch[
                    "video_lengths"
                ]
            )

            predictions.extend(
                decoded
            )

            references.extend(
                batch["texts"]
            )

    avg_loss = (
        running_loss
        / max(
            1,
            steps
        )
    )

    cer = (
        character_error_rate(
            predictions,
            references
        )
        if references
        else np.nan
    )

    return (
        avg_loss,
        cer,
        predictions,
        references
    )


print("Training functions ready")

In [ ]:
print("train_one_epoch exists:", "train_one_epoch" in globals())

## Cell 17 — Sanity Overfit Test (แนะนำให้ทำก่อน)

รอบนี้ใช้เพียง 16 training clips เพื่อเช็กว่า model สามารถเรียนข้อมูลเล็ก ๆ ได้

ถ้า loss ไม่ลดเลย แปลว่า pipeline/model มีปัญหาก่อน Full Training

In [ ]:
from torch.utils.data import Subset

SANITY_SAMPLES = min(
    16,
    len(train_dataset)
)

sanity_subset = Subset(
    train_dataset,
    list(
        range(
            SANITY_SAMPLES
        )
    )
)

sanity_loader = DataLoader(
    sanity_subset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn,
)

# fresh model สำหรับ sanity
sanity_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    feature_dim=128,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

sanity_optimizer = torch.optim.AdamW(
    sanity_model.parameters(),
    lr=5e-4,
)

sanity_scaler = torch.amp.GradScaler(
    "cuda"
)

# ใช้ global scaler ชั่วคราวกับ helper เดิม
_original_scaler = scaler
scaler = sanity_scaler

SANITY_EPOCHS = 8

for epoch in range(
    1,
    SANITY_EPOCHS + 1
):

    loss = train_one_epoch(
        sanity_model,
        sanity_loader,
        sanity_optimizer
    )

    print(
        f"Sanity Epoch "
        f"{epoch:02d}/{SANITY_EPOCHS} "
        f"| loss={loss:.4f}"
    )

scaler = _original_scaler

print("Sanity test finished")

## Cell 18 — Reset Main Model ก่อน Full Training

Sanity model แยกออกจาก main model อยู่แล้ว แต่ Cell นี้ reset optimizer state ให้ชัดเจน

In [ ]:
model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

scaler = torch.amp.GradScaler(
    "cuda"
)

print("Main model reset")

In [ ]:
check_names = [
    "DEVICE",
    "train_dataset",
    "collate_fn",
    "LipReadingBaseline",
    "VOCAB_SIZE",
    "BLANK_ID",
    "GRU_HIDDEN",
    "GRU_LAYERS",
    "DROPOUT",
    "train_one_epoch",
]

print("=" * 60)

for name in check_names:
    print(
        f"{name:25s}: "
        f"{'OK' if name in globals() else 'MISSING'}"
    )

print("=" * 60)

In [ ]:
# ============================================================
# SANITY OVERFIT TEST V2
# ทดสอบว่าโมเดลสามารถจำข้อมูล Train เพียง 8 clips ได้หรือไม่
# ============================================================

import torch
from torch.utils.data import DataLoader, Subset

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
SANITY_SAMPLES = 8
SANITY_EPOCHS = 100
SANITY_BATCH_SIZE = 1
SANITY_LR = 5e-4

print("=" * 70)
print("SANITY OVERFIT TEST V2")
print("=" * 70)
print("Device :", DEVICE)
print("Samples:", SANITY_SAMPLES)
print("Epochs :", SANITY_EPOCHS)


# ------------------------------------------------------------
# 1. ใช้ Train เพียง 8 clips
# ------------------------------------------------------------
sanity_subset = Subset(
    train_dataset,
    list(range(SANITY_SAMPLES))
)

sanity_loader = DataLoader(
    sanity_subset,
    batch_size=SANITY_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

print("Sanity clips:", len(sanity_subset))


# ------------------------------------------------------------
# 2. สร้าง Model ใหม่
# ไม่ยุ่งกับ V1 / best_model.pt เดิม
# ------------------------------------------------------------
sanity_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    feature_dim=128,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

sanity_optimizer = torch.optim.AdamW(
    sanity_model.parameters(),
    lr=SANITY_LR,
    weight_decay=0.0
)

# scaler ใหม่สำหรับ sanity model
sanity_scaler = torch.amp.GradScaler("cuda")

# train_one_epoch เดิมใน notebook ใช้ตัวแปร scaler แบบ global
_original_scaler = scaler
scaler = sanity_scaler


# ------------------------------------------------------------
# 3. TRAIN
# ------------------------------------------------------------
print("\nStarting sanity training...\n")

sanity_history = []

try:

    for epoch in range(1, SANITY_EPOCHS + 1):

        loss = train_one_epoch(
            sanity_model,
            sanity_loader,
            sanity_optimizer
        )

        sanity_history.append(loss)

        # แสดงทุก 5 epochs
        if epoch == 1 or epoch % 5 == 0:

            print(
                f"Sanity Epoch "
                f"{epoch:03d}/{SANITY_EPOCHS} "
                f"| loss={loss:.4f}"
            )

        # ถ้า loss ต่ำมากแล้ว ไม่จำเป็นต้องฝึกต่อ
        if loss < 0.10:

            print("\nSanity loss < 0.10")
            print("Model can strongly overfit the tiny dataset.")

            break

finally:

    # คืน scaler เดิมเสมอ
    scaler = _original_scaler


# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("SANITY TRAINING FINISHED")
print("=" * 70)

print("First loss:", round(sanity_history[0], 4))
print("Final loss:", round(sanity_history[-1], 4))

print(
    "Loss reduction:",
    round(
        sanity_history[0] - sanity_history[-1],
        4
    )
)

print("=" * 70)

In [ ]:
# ============================================================
# SANITY EVALUATION
# CER + Blank Ratio + REF / PRED
# ============================================================

import torch


# ------------------------------------------------------------
# CTC GREEDY DECODER
# ใช้ charset mapping จาก notebook
# ------------------------------------------------------------

# หา mapping ที่ notebook มีอยู่
print("Checking character mapping...")

if "id_to_char" in globals():
    sanity_id_to_char = id_to_char

elif "idx_to_char" in globals():
    sanity_id_to_char = idx_to_char

elif "itos" in globals():
    sanity_id_to_char = {
        i: ch for i, ch in enumerate(itos)
    }

else:
    sanity_id_to_char = None


print(
    "Character mapping:",
    "FOUND" if sanity_id_to_char is not None else "NOT FOUND"
)

if sanity_id_to_char is None:
    raise RuntimeError(
        "ไม่พบ id_to_char / idx_to_char / itos "
        "กรุณาหยุดและส่งชื่อ charset variables มา"
    )


# ------------------------------------------------------------
# Edit Distance
# ------------------------------------------------------------

def sanity_edit_distance(ref, hyp):

    m = len(ref)
    n = len(hyp)

    dp = [
        [0] * (n + 1)
        for _ in range(m + 1)
    ]

    for i in range(m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):

        for j in range(1, n + 1):

            cost = (
                0
                if ref[i - 1] == hyp[j - 1]
                else 1
            )

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[m][n]


# ------------------------------------------------------------
# CTC Decode
# ------------------------------------------------------------

def sanity_decode(pred_ids, length):

    ids = pred_ids[:length].detach().cpu().tolist()

    decoded = []

    previous = None

    for idx in ids:

        # CTC collapse repeated tokens
        if idx != previous:

            # remove blank
            if idx != BLANK_ID:

                if idx in sanity_id_to_char:
                    decoded.append(
                        sanity_id_to_char[idx]
                    )

        previous = idx

    return "".join(decoded)


# ------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------

sanity_model.eval()

references = []
predictions = []

total_chars = 0
total_errors = 0

total_steps = 0
blank_steps = 0


with torch.no_grad():

    for batch in sanity_loader:

        videos = batch["video"].to(DEVICE)

        lengths = batch["video_lengths"]

        logits = sanity_model(
            videos,
            lengths
        )

        pred_ids = logits.argmax(dim=-1)

        # ----------------------------------------
        # แต่ละ sample
        # ----------------------------------------

        for i in range(pred_ids.size(0)):

            length = int(
                lengths[i].item()
            )

            valid_ids = pred_ids[i, :length]

            # Blank statistics
            total_steps += valid_ids.numel()

            blank_steps += (
                valid_ids == BLANK_ID
            ).sum().item()

            # Decode
            pred_text = sanity_decode(
                pred_ids[i],
                length
            )

            ref_text = batch["texts"][i]

            references.append(ref_text)
            predictions.append(pred_text)

            # CER
            errors = sanity_edit_distance(
                ref_text,
                pred_text
            )

            total_errors += errors
            total_chars += len(ref_text)


# ------------------------------------------------------------
# Final Metrics
# ------------------------------------------------------------

sanity_cer = (
    total_errors / total_chars
    if total_chars > 0
    else 0
)

sanity_blank_ratio = (
    blank_steps / total_steps
    if total_steps > 0
    else 0
)


print("\n" + "=" * 70)
print("SANITY EVALUATION RESULT")
print("=" * 70)

print(f"CER         : {sanity_cer:.4f}")
print(
    f"Blank Ratio : {sanity_blank_ratio:.4f} "
    f"({sanity_blank_ratio * 100:.2f}%)"
)

print("=" * 70)


# ------------------------------------------------------------
# แสดงทั้ง 8 clips
# ------------------------------------------------------------

for i, (ref, pred) in enumerate(
    zip(references, predictions),
    start=1
):

    sample_cer = (
        sanity_edit_distance(ref, pred) / len(ref)
        if len(ref) > 0
        else 0
    )

    print(f"\nSample {i}")
    print("REF :", ref)
    print("PRED:", pred)
    print(f"CER : {sample_cer:.4f}")
    print("-" * 70)

In [ ]:
# ============================================================
# CHECK CTC COMPATIBILITY — SANITY 8 CLIPS
# ============================================================

print("=" * 90)
print("SANITY CTC LENGTH CHECK")
print("=" * 90)

bad_count = 0

for i in range(len(sanity_subset)):

    sample = sanity_subset[i]

    video = sample["video"]
    text = sample["text"]
    target = sample["target"]

    frames = video.shape[0]
    target_len = len(target)

    ratio = frames / max(target_len, 1)

    status = "OK"

    if frames < target_len:
        status = "INVALID"
        bad_count += 1

    elif ratio < 1.5:
        status = "VERY TIGHT"

    elif ratio < 2.0:
        status = "TIGHT"

    print(
        f"{i+1:02d} | "
        f"frames={frames:3d} | "
        f"target={target_len:3d} | "
        f"ratio={ratio:5.2f} | "
        f"{status}"
    )

    print("TEXT:", text)
    print("-" * 90)

print()
print("Invalid clips:", bad_count)

In [ ]:
# ============================================================
# TRUE CTC ALIGNMENT CHECK
# ตรวจ target length + adjacent repeated labels
# ============================================================

print("=" * 100)
print("TRUE CTC ALIGNMENT CHECK — SANITY 8 CLIPS")
print("=" * 100)

invalid = 0
tight = 0

for i in range(len(sanity_subset)):

    sample = sanity_subset[i]

    video = sample["video"]
    target = sample["target"]
    text = sample["text"]

    frames = int(video.shape[0])

    # target อาจเป็น tensor
    if torch.is_tensor(target):
        target_ids = target.detach().cpu().tolist()
    else:
        target_ids = list(target)

    target_len = len(target_ids)

    # จำนวน label ที่ซ้ำติดกัน
    repeats = sum(
        1
        for j in range(1, target_len)
        if target_ids[j] == target_ids[j - 1]
    )

    # จำนวน timestep ขั้นต่ำที่ CTC ต้องการจริง
    min_ctc_steps = target_len + repeats

    margin = frames - min_ctc_steps
    ratio = frames / max(min_ctc_steps, 1)

    if frames < min_ctc_steps:
        status = "INVALID"
        invalid += 1

    elif ratio < 1.5:
        status = "VERY TIGHT"
        tight += 1

    elif ratio < 2.0:
        status = "TIGHT"
        tight += 1

    else:
        status = "OK"

    print(
        f"{i+1:02d} | "
        f"frames={frames:3d} | "
        f"target={target_len:3d} | "
        f"repeats={repeats:2d} | "
        f"min_ctc={min_ctc_steps:3d} | "
        f"margin={margin:3d} | "
        f"ratio={ratio:5.2f} | "
        f"{status}"
    )

    print("TEXT:", text)
    print("-" * 100)

print()
print("Invalid :", invalid)
print("Tight   :", tight)

In [ ]:
# ============================================================
# CHECK TARGET ENCODING
# ดูว่า text -> target ids -> text กลับมาได้ตรงเดิมไหม
# ============================================================

print("=" * 100)
print("TARGET ENCODING CHECK — SANITY 8 CLIPS")
print("=" * 100)

for i in range(len(sanity_subset)):

    sample = sanity_subset[i]

    text = sample["text"]
    target = sample["target"]

    if torch.is_tensor(target):
        target_ids = target.detach().cpu().tolist()
    else:
        target_ids = list(target)

    # Decode target IDs กลับเป็นข้อความ
    decoded = "".join(
        sanity_id_to_char[idx]
        for idx in target_ids
        if idx in sanity_id_to_char
    )

    match = text == decoded

    print(f"\nSample {i+1}")
    print("TEXT    :", repr(text))
    print("DECODED :", repr(decoded))
    print("MATCH   :", match)
    print("TEXT LEN:", len(text))
    print("IDS LEN :", len(target_ids))
    print("TARGET  :", target_ids)
    print("-" * 100)

In [ ]:
# ============================================================
# ONE-CLIP OVERFIT TEST
# ทดสอบขั้นเด็ดขาดว่า model สามารถจำคลิปเดียวได้หรือไม่
# ============================================================

import torch
from torch.utils.data import DataLoader, Subset

ONE_CLIP_INDEX = 4      # Sample 5 (index เริ่มจาก 0)
ONE_EPOCHS = 300
ONE_LR = 1e-3

# ------------------------------------------------------------
# Dataset = เพียง 1 clip
# ------------------------------------------------------------

one_subset = Subset(
    train_dataset,
    [ONE_CLIP_INDEX]
)

one_loader = DataLoader(
    one_subset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

sample = one_subset[0]

print("=" * 70)
print("ONE-CLIP OVERFIT TEST")
print("=" * 70)

print("TEXT   :", sample["text"])
print("Frames :", sample["video"].shape[0])
print("Target :", len(sample["target"]))


# ------------------------------------------------------------
# สร้าง Model ใหม่
#
# dropout = 0 เพื่อให้ง่ายต่อการจำ clip เดียว
# ------------------------------------------------------------

one_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    feature_dim=128,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=0.0
).to(DEVICE)

one_optimizer = torch.optim.AdamW(
    one_model.parameters(),
    lr=ONE_LR,
    weight_decay=0.0
)

one_criterion = torch.nn.CTCLoss(
    blank=BLANK_ID,
    zero_infinity=True
)

# ============================================================
# DIAGNOSE ONE-CLIP:
# TRAIN MODE vs EVAL MODE
# ============================================================

import torch

batch = next(iter(one_loader))

videos = batch["video"].to(DEVICE)
video_lengths = batch["video_lengths"]

targets = batch["targets"].to(DEVICE)
target_lengths = batch["target_lengths"]

ref_text = batch["texts"][0]


def inspect_one_clip(model, mode_name):

    if mode_name == "TRAIN":
        model.train()
    else:
        model.eval()

    with torch.no_grad():

        logits = model(
            videos,
            video_lengths
        )

        # -------------------------------
        # CTC Loss
        # -------------------------------

        log_probs = logits.log_softmax(
            dim=-1
        ).transpose(0, 1)

        loss = one_criterion(
            log_probs,
            targets,
            video_lengths,
            target_lengths
        )

        # -------------------------------
        # Argmax Decode
        # -------------------------------

        pred_ids = logits.argmax(dim=-1)

        length = int(
            video_lengths[0].item()
        )

        valid_ids = pred_ids[0, :length]

        blank_ratio = (
            (valid_ids == BLANK_ID)
            .float()
            .mean()
            .item()
        )

        pred_text = sanity_decode(
            pred_ids[0],
            length
        )

    print("=" * 70)
    print(mode_name, "MODE")
    print("=" * 70)

    print(f"CTC Loss    : {loss.item():.6f}")
    print(f"Blank Ratio : {blank_ratio*100:.2f}%")

    print("REF :", ref_text)
    print("PRED:", pred_text)

    print()


inspect_one_clip(
    one_model,
    "TRAIN"
)

inspect_one_clip(
    one_model,
    "EVAL"
)
# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

for epoch in range(1, ONE_EPOCHS + 1):

    one_model.train()

    batch = next(iter(one_loader))

    videos = batch["video"].to(DEVICE)
    video_lengths = batch["video_lengths"]

    targets = batch["targets"].to(DEVICE)
    target_lengths = batch["target_lengths"]

    one_optimizer.zero_grad()

    logits = one_model(
        videos,
        video_lengths
    )

    log_probs = logits.log_softmax(
        dim=-1
    ).transpose(0, 1)

    loss = one_criterion(
        log_probs,
        targets,
        video_lengths,
        target_lengths
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        one_model.parameters(),
        5.0
    )

    one_optimizer.step()


    # --------------------------------------------------------
    # Evaluate ทุก 10 epochs
    # --------------------------------------------------------

    if epoch == 1 or epoch % 10 == 0:

        one_model.eval()

        with torch.no_grad():

            logits_eval = one_model(
                videos,
                video_lengths
            )

            pred_ids = logits_eval.argmax(dim=-1)

            length = int(video_lengths[0].item())

            valid_ids = pred_ids[0, :length]

            blank_ratio = (
                (valid_ids == BLANK_ID)
                .float()
                .mean()
                .item()
            )

            pred_text = sanity_decode(
                pred_ids[0],
                length
            )

        print(
            f"Epoch {epoch:03d}/{ONE_EPOCHS} "
            f"| loss={loss.item():.4f} "
            f"| blank={blank_ratio*100:.2f}%"
        )

        print("REF :", batch["texts"][0])
        print("PRED:", pred_text)
        print("-" * 70)

        # ถ้าจำได้ตรงแล้ว หยุดได้เลย
        if pred_text == batch["texts"][0]:

            print("\n🎉 ONE-CLIP OVERFIT PASSED")
            print("Model memorized the clip exactly.")

            break

In [ ]:
# ============================================================
# PROGRESSIVE OVERFIT TEST — 2 CLIPS
# ============================================================

import torch
from torch.utils.data import DataLoader, Subset

NUM_CLIPS = 2
TEST_EPOCHS = 300
TEST_LR = 1e-3

# เลือก clip ที่ length ค่อนข้างดี
# Sample 5 + Sample 4 จาก sanity set เดิม
selected_indices = [4, 3]

test_subset = Subset(
    train_dataset,
    selected_indices
)

test_loader = DataLoader(
    test_subset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

test_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    feature_dim=128,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=0.0
).to(DEVICE)

optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=TEST_LR,
    weight_decay=0.0
)

criterion = torch.nn.CTCLoss(
    blank=BLANK_ID,
    zero_infinity=True
)

print("=" * 70)
print("2-CLIP OVERFIT TEST")
print("=" * 70)

for i in range(len(test_subset)):
    sample = test_subset[i]
    print(
        f"{i+1}. "
        f"frames={sample['video'].shape[0]} | "
        f"target={len(sample['target'])}"
    )
    print("   ", sample["text"])

print("=" * 70)


for epoch in range(1, TEST_EPOCHS + 1):

    test_model.train()

    epoch_loss = 0.0

    for batch in test_loader:

        videos = batch["video"].to(DEVICE)
        video_lengths = batch["video_lengths"]

        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"]

        optimizer.zero_grad()

        logits = test_model(
            videos,
            video_lengths
        )

        log_probs = logits.log_softmax(
            dim=-1
        ).transpose(0, 1)

        loss = criterion(
            log_probs,
            targets,
            video_lengths,
            target_lengths
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            test_model.parameters(),
            5.0
        )

        optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(test_loader)

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------
    if epoch == 1 or epoch % 20 == 0:

        test_model.eval()

        exact = 0
        total = 0

        print(
            f"\nEpoch {epoch:03d}/{TEST_EPOCHS} "
            f"| loss={epoch_loss:.4f}"
        )

        with torch.no_grad():

            for batch in test_loader:

                videos = batch["video"].to(DEVICE)
                lengths = batch["video_lengths"]

                logits = test_model(
                    videos,
                    lengths
                )

                pred_ids = logits.argmax(dim=-1)

                for b in range(pred_ids.size(0)):

                    length = int(
                        lengths[b].item()
                    )

                    pred = sanity_decode(
                        pred_ids[b],
                        length
                    )

                    ref = batch["texts"][b]

                    print("REF :", ref)
                    print("PRED:", pred)

                    if pred == ref:
                        exact += 1

                    total += 1

        print(
            f"Exact Match: {exact}/{total}"
        )
        print("-" * 70)

        # จำได้ครบทั้ง 2 clips
        if exact == total:

            print(
                "\n🎉 2-CLIP OVERFIT PASSED"
            )

            break

In [ ]:
# ============================================================
# FINAL 2-CLIP EVALUATION
# ============================================================

test_model.eval()

total_chars = 0
total_errors = 0
total_steps = 0
blank_steps = 0
exact = 0
total = 0

print("=" * 80)
print("FINAL 2-CLIP RESULT")
print("=" * 80)

with torch.no_grad():

    for batch in test_loader:

        videos = batch["video"].to(DEVICE)
        lengths = batch["video_lengths"]

        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"]

        logits = test_model(
            videos,
            lengths
        )

        # CTC loss
        log_probs = logits.log_softmax(dim=-1).transpose(0, 1)

        loss = criterion(
            log_probs,
            targets,
            lengths,
            target_lengths
        )

        pred_ids = logits.argmax(dim=-1)

        for b in range(pred_ids.size(0)):

            length = int(lengths[b].item())

            valid = pred_ids[b, :length]

            total_steps += valid.numel()
            blank_steps += (
                valid == BLANK_ID
            ).sum().item()

            pred = sanity_decode(
                pred_ids[b],
                length
            )

            ref = batch["texts"][b]

            err = sanity_edit_distance(
                ref,
                pred
            )

            cer = err / max(len(ref), 1)

            total_errors += err
            total_chars += len(ref)

            if pred == ref:
                exact += 1

            total += 1

            print("REF :", ref)
            print("PRED:", pred)
            print(f"CER : {cer:.4f}")
            print("-" * 80)


overall_cer = total_errors / max(total_chars, 1)
blank_ratio = blank_steps / max(total_steps, 1)

print()
print("=" * 80)
print(f"Exact Match : {exact}/{total}")
print(f"Overall CER : {overall_cer:.4f}")
print(f"Blank Ratio : {blank_ratio:.4f} ({blank_ratio*100:.2f}%)")
print("=" * 80)

In [ ]:
# ============================================================
# CONTINUE 2-CLIP OVERFIT
# Fine-tune ต่อจาก model เดิม
# ============================================================

CONTINUE_EPOCHS = 200

fine_optimizer = torch.optim.AdamW(
    test_model.parameters(),
    lr=2e-4,
    weight_decay=0.0
)

for epoch in range(1, CONTINUE_EPOCHS + 1):

    test_model.train()
    epoch_loss = 0.0

    for batch in test_loader:

        videos = batch["video"].to(DEVICE)
        lengths = batch["video_lengths"]

        targets = batch["targets"].to(DEVICE)
        target_lengths = batch["target_lengths"]

        fine_optimizer.zero_grad()

        logits = test_model(
            videos,
            lengths
        )

        log_probs = logits.log_softmax(
            dim=-1
        ).transpose(0, 1)

        loss = criterion(
            log_probs,
            targets,
            lengths,
            target_lengths
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            test_model.parameters(),
            5.0
        )

        fine_optimizer.step()

        epoch_loss += loss.item()

    epoch_loss /= len(test_loader)

    # --------------------------------------------------------
    # Evaluate ทุก 20 epochs
    # --------------------------------------------------------
    if epoch == 1 or epoch % 20 == 0:

        test_model.eval()

        exact = 0
        total = 0

        print(
            f"\nFine Epoch {epoch:03d}/{CONTINUE_EPOCHS}"
            f" | loss={epoch_loss:.6f}"
        )

        with torch.no_grad():

            for batch in test_loader:

                videos = batch["video"].to(DEVICE)
                lengths = batch["video_lengths"]

                logits = test_model(
                    videos,
                    lengths
                )

                pred_ids = logits.argmax(dim=-1)

                for b in range(pred_ids.size(0)):

                    length = int(lengths[b].item())

                    pred = sanity_decode(
                        pred_ids[b],
                        length
                    )

                    ref = batch["texts"][b]

                    print("REF :", ref)
                    print("PRED:", pred)

                    if pred == ref:
                        exact += 1

                    total += 1

        print(f"Exact Match: {exact}/{total}")
        print("-" * 70)

        if exact == total:

            print("\n🎉 2-CLIP EXACT OVERFIT PASSED")
            break

## Cell 19 — Full Training 🔥

เซฟ best model ตาม Validation Loss

Early stopping หลัง val loss ไม่ดีขึ้นตาม `PATIENCE`

In [ ]:
import time

BEST_MODEL_PATH = (
    MODEL_ROOT
    / "best_model.pt"
)

LAST_MODEL_PATH = (
    MODEL_ROOT
    / "last_model.pt"
)

HISTORY_CSV = (
    MODEL_ROOT
    / "training_history.csv"
)

history = []

best_val_loss = float(
    "inf"
)

epochs_without_improvement = 0

training_start = time.time()

for epoch in range(
    1,
    EPOCHS + 1
):

    epoch_start = time.time()

    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    (
        val_loss,
        val_cer,
        val_preds,
        val_refs
    ) = evaluate(
        model,
        val_loader
    )

    scheduler.step(
        val_loss
    )

    lr = optimizer.param_groups[
        0
    ]["lr"]

    elapsed = (
        time.time()
        - epoch_start
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_cer": val_cer,
        "lr": lr,
        "epoch_seconds": elapsed,
    }

    history.append(
        row
    )

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train={train_loss:.4f} | "
        f"val={val_loss:.4f} | "
        f"CER={val_cer:.4f} | "
        f"lr={lr:.2e} | "
        f"{elapsed/60:.1f} min"
    )

    # preview predictions
    print("\nExamples:")

    for pred, ref in list(
        zip(
            val_preds,
            val_refs
        )
    )[:3]:

        print(
            "REF :",
            ref
        )

        print(
            "PRED:",
            pred
        )

        print("-" * 50)

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        epochs_without_improvement = 0

        torch.save(
            {
                "model_state_dict":
                    model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "epoch":
                    epoch,
                "best_val_loss":
                    best_val_loss,
                "vocab_size":
                    VOCAB_SIZE,
                "itos":
                    itos,
                "config": {
                    "image_size":
                        IMAGE_SIZE,
                    "gru_hidden":
                        GRU_HIDDEN,
                    "gru_layers":
                        GRU_LAYERS,
                    "dropout":
                        DROPOUT,
                },
            },
            BEST_MODEL_PATH
        )

        print(
            "Saved BEST model:",
            BEST_MODEL_PATH
        )

    else:

        epochs_without_improvement += 1

    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    if (
        epochs_without_improvement
        >= PATIENCE
    ):

        print(
            "\nEarly stopping"
        )

        break


torch.save(
    {
        "model_state_dict":
            model.state_dict(),
        "epoch":
            history[-1]["epoch"],
        "vocab_size":
            VOCAB_SIZE,
        "itos":
            itos,
    },
    LAST_MODEL_PATH
)

print("\nTraining complete")
print(
    "Total training time:",
    round(
        (
            time.time()
            - training_start
        )
        / 60,
        2
    ),
    "minutes"
)

print(
    "Best model:",
    BEST_MODEL_PATH
)

print(
    "History:",
    HISTORY_CSV
)

## Cell 20 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.read_csv(
    HISTORY_CSV
)

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train Loss"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Val Loss"
)

plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.title("Training Curve")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(
    figsize=(8, 5)
)

plt.plot(
    history_df["epoch"],
    history_df["val_cer"]
)

plt.xlabel("Epoch")
plt.ylabel("CER")
plt.title("Validation Character Error Rate")
plt.grid(True)
plt.show()

## Cell 21 — Load Best Model + Test Evaluation

In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

best_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    hidden_dim=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

best_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

(
    test_loss,
    test_cer,
    test_preds,
    test_refs
) = evaluate(
    best_model,
    test_loader
)

print("=" * 70)
print("TEST RESULT")
print("=" * 70)

print(
    "Test Loss:",
    round(
        test_loss,
        4
    )
)

print(
    "Test CER :",
    round(
        test_cer,
        4
    )
)

## Cell 22 — Test Predictions

In [ ]:
for i, (
    pred,
    ref
) in enumerate(
    zip(
        test_preds,
        test_refs
    )
):

    print(
        f"[{i:03d}]"
    )

    print(
        "REF :",
        ref
    )

    print(
        "PRED:",
        pred
    )

    print(
        "-" * 70
    )

    if i >= 19:
        break

In [ ]:
# ============================================================
# CTC Blank Collapse Diagnostic
# ============================================================

import torch

model.eval()

# ดูว่า model อยู่ device ไหน
DEVICE = next(model.parameters()).device
print("Model device:", DEVICE)

total_steps = 0
blank_steps = 0
nonblank_steps = 0

with torch.no_grad():

    for batch in test_loader:

        # ----------------------------------------
        # ย้าย VIDEO จาก CPU -> device เดียวกับ model
        # ----------------------------------------
        videos = batch["video"].to(DEVICE)

        # video_lengths ใช้สำหรับบอกความยาว sequence
        lengths = batch["video_lengths"]

        # Forward
        logits = model(videos, lengths)

        # เลือก class ที่มี probability สูงสุด
        pred_ids = logits.argmax(dim=-1)

        # ----------------------------------------
        # นับเฉพาะ frame จริง ไม่รวม padding
        # ----------------------------------------
        for i in range(pred_ids.size(0)):

            length = int(lengths[i].item())

            valid_pred = pred_ids[i, :length]

            total_steps += valid_pred.numel()

            blank_steps += (
                valid_pred == BLANK_ID
            ).sum().item()

            nonblank_steps += (
                valid_pred != BLANK_ID
            ).sum().item()


# ============================================================
# RESULT
# ============================================================

blank_ratio = blank_steps / total_steps
nonblank_ratio = nonblank_steps / total_steps

print("\n" + "=" * 60)
print("CTC BLANK DIAGNOSTIC")
print("=" * 60)

print(f"Total valid timesteps : {total_steps:,}")
print(f"Blank timesteps       : {blank_steps:,}")
print(f"Non-blank timesteps   : {nonblank_steps:,}")

print("-" * 60)

print(f"Blank ratio           : {blank_ratio:.4f} ({blank_ratio*100:.2f}%)")
print(f"Non-blank ratio       : {nonblank_ratio:.4f} ({nonblank_ratio*100:.2f}%)")

print("=" * 60)

In [ ]:
batch = next(iter(test_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    print("Batch keys:", batch.keys())
    for key, value in batch.items():
        if hasattr(value, "shape"):
            print(f"{key:20s} shape = {value.shape}")
        else:
            print(f"{key:20s} type = {type(value)}")
else:
    print(batch)

In [ ]:
# ============================================================
# SANITY OVERFIT TEST — 8 CLIPS
# ตรวจว่า Model + CTC + Dataset pipeline สามารถ "จำ" ข้อมูลเล็ก ๆ ได้หรือไม่
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm
import numpy as np


# ------------------------------------------------------------
# 1. CONFIG
# ------------------------------------------------------------

SANITY_SAMPLES = 8
SANITY_EPOCHS = 100
SANITY_BATCH_SIZE = 1
SANITY_LR = 5e-4

print("=" * 70)
print("SANITY OVERFIT TEST")
print("=" * 70)

print("Device          :", DEVICE)
print("Samples         :", SANITY_SAMPLES)
print("Epochs          :", SANITY_EPOCHS)
print("Batch size      :", SANITY_BATCH_SIZE)
print("Learning rate   :", SANITY_LR)


# ------------------------------------------------------------
# 2. เลือก 8 clips จาก Train Dataset
# ------------------------------------------------------------

sanity_indices = list(range(min(SANITY_SAMPLES, len(train_dataset))))

sanity_subset = Subset(
    train_dataset,
    sanity_indices
)

sanity_loader = DataLoader(
    sanity_subset,
    batch_size=SANITY_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_fn
)

print("\nSanity dataset size:", len(sanity_subset))


# ------------------------------------------------------------
# 3. สร้าง MODEL ใหม่
#
# สำคัญ:
# ไม่ใช้ model V1 เดิม
# เพื่อดูว่า architecture สามารถเรียนรู้ 8 clips จากศูนย์ได้หรือไม่
# ------------------------------------------------------------

sanity_model = LipReadingBaseline(
    vocab_size=VOCAB_SIZE,
    feature_dim=128,
    hidden_size=GRU_HIDDEN,
    num_layers=GRU_LAYERS,
    dropout=DROPOUT
).to(DEVICE)

print("Model device:", next(sanity_model.parameters()).device)


# ------------------------------------------------------------
# 4. CTC LOSS
# ------------------------------------------------------------

sanity_criterion = nn.CTCLoss(
    blank=BLANK_ID,
    reduction="mean",
    zero_infinity=True
)


# ------------------------------------------------------------
# 5. OPTIMIZER
# ------------------------------------------------------------

sanity_optimizer = torch.optim.AdamW(
    sanity_model.parameters(),
    lr=SANITY_LR,
    weight_decay=0.0
)


# ------------------------------------------------------------
# 6. Greedy CTC Decoder
# ------------------------------------------------------------

def sanity_ctc_decode(logits, lengths):

    pred_ids = logits.argmax(dim=-1)

    predictions = []

    for i in range(pred_ids.size(0)):

        length = int(lengths[i].item())

        sequence = pred_ids[i, :length].detach().cpu().tolist()

        decoded = []

        previous = None

        for idx in sequence:

            # CTC:
            # 1. ลบตัวซ้ำติดกัน
            # 2. ลบ blank
            if idx != previous:

                if idx != BLANK_ID:
                    decoded.append(idx)

            previous = idx

        # ใช้ id_to_char จาก notebook เดิม
        text = "".join(
            id_to_char[idx]
            for idx in decoded
            if idx in id_to_char
        )

        predictions.append(text)

    return predictions


# ------------------------------------------------------------
# 7. CER FUNCTION
# ------------------------------------------------------------

def edit_distance(ref, hyp):

    m = len(ref)
    n = len(hyp)

    dp = np.zeros((m + 1, n + 1), dtype=np.int32)

    for i in range(m + 1):
        dp[i, 0] = i

    for j in range(n + 1):
        dp[0, j] = j

    for i in range(1, m + 1):

        for j in range(1, n + 1):

            cost = 0 if ref[i - 1] == hyp[j - 1] else 1

            dp[i, j] = min(
                dp[i - 1, j] + 1,
                dp[i, j - 1] + 1,
                dp[i - 1, j - 1] + cost
            )

    return dp[m, n]


def calculate_cer(refs, hyps):

    total_errors = 0
    total_chars = 0

    for ref, hyp in zip(refs, hyps):

        total_errors += edit_distance(ref, hyp)
        total_chars += len(ref)

    if total_chars == 0:
        return 0.0

    return total_errors / total_chars


# ------------------------------------------------------------
# 8. TRAIN ONE SANITY EPOCH
# ------------------------------------------------------------

def train_sanity_epoch():

    sanity_model.train()

    total_loss = 0.0

    for batch in sanity_loader:

        # -------------------------------
        # Input
        # -------------------------------

        videos = batch["video"].to(
            DEVICE,
            non_blocking=True
        )

        video_lengths = batch["video_lengths"]

        targets = batch["targets"].to(
            DEVICE
        )

        target_lengths = batch["target_lengths"]

        # -------------------------------
        # Reset gradients
        # -------------------------------

        sanity_optimizer.zero_grad(
            set_to_none=True
        )

        # -------------------------------
        # Forward
        # -------------------------------

        logits = sanity_model(
            videos,
            video_lengths
        )

        # CTC ต้องการ:
        #
        # [T, B, C]
        #
        log_probs = logits.log_softmax(
            dim=-1
        ).transpose(0, 1)

        # -------------------------------
        # CTC Loss
        # -------------------------------

        loss = sanity_criterion(
            log_probs,
            targets,
            video_lengths,
            target_lengths
        )

        # -------------------------------
        # Backprop
        # -------------------------------

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            sanity_model.parameters(),
            5.0
        )

        sanity_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(sanity_loader)


# ------------------------------------------------------------
# 9. EVALUATE SANITY SET
# ------------------------------------------------------------

def evaluate_sanity():

    sanity_model.eval()

    refs = []
    preds = []

    blank_count = 0
    total_steps = 0

    with torch.no_grad():

        for batch in sanity_loader:

            videos = batch["video"].to(DEVICE)

            lengths = batch["video_lengths"]

            logits = sanity_model(
                videos,
                lengths
            )

            pred_ids = logits.argmax(
                dim=-1
            )

            # -------------------------------
            # Blank Ratio
            # -------------------------------

            for i in range(pred_ids.size(0)):

                length = int(
                    lengths[i].item()
                )

                valid = pred_ids[
                    i,
                    :length
                ]

                blank_count += (
                    valid == BLANK_ID
                ).sum().item()

                total_steps += valid.numel()

            # -------------------------------
            # Decode
            # -------------------------------

            batch_predictions = sanity_ctc_decode(
                logits,
                lengths
            )

            refs.extend(
                batch["texts"]
            )

            preds.extend(
                batch_predictions
            )

    cer = calculate_cer(
        refs,
        preds
    )

    blank_ratio = (
        blank_count / total_steps
        if total_steps > 0
        else 0
    )

    return cer, blank_ratio, refs, preds


# ------------------------------------------------------------
# 10. TRAIN SANITY MODEL
# ------------------------------------------------------------

history = []

print("\n")
print("=" * 70)
print("START SANITY TRAINING")
print("=" * 70)


for epoch in range(1, SANITY_EPOCHS + 1):

    loss = train_sanity_epoch()

    cer, blank_ratio, refs, preds = evaluate_sanity()

    history.append({
        "epoch": epoch,
        "loss": loss,
        "cer": cer,
        "blank_ratio": blank_ratio
    })

    # --------------------------------------------------------
    # แสดงทุก 5 Epoch
    # --------------------------------------------------------

    if (
        epoch == 1
        or epoch % 5 == 0
        or epoch == SANITY_EPOCHS
    ):

        print(
            f"\nEpoch {epoch:03d}/{SANITY_EPOCHS}"
            f" | Loss={loss:.4f}"
            f" | CER={cer:.4f}"
            f" | Blank={blank_ratio*100:.2f}%"
        )

        print("-" * 70)

        # แสดงตัวอย่าง 3 ตัว
        for i in range(min(3, len(refs))):

            print("REF :", refs[i])
            print("PRED:", preds[i])
            print("-" * 70)

    # --------------------------------------------------------
    # ถ้าจำข้อมูลได้เกือบหมดแล้ว หยุดได้
    # --------------------------------------------------------

    if cer < 0.05:

        print("\n🎉 SANITY TEST PASSED")
        print(
            f"Model overfit 8 clips at epoch {epoch}"
        )

        break


# ------------------------------------------------------------
# 11. FINAL RESULT
# ------------------------------------------------------------

cer, blank_ratio, refs, preds = evaluate_sanity()

print("\n")
print("=" * 70)
print("FINAL SANITY RESULT")
print("=" * 70)

print(f"CER         : {cer:.4f}")
print(
    f"Blank Ratio : {blank_ratio:.4f} "
    f"({blank_ratio*100:.2f}%)"
)

print("=" * 70)


# ------------------------------------------------------------
# 12. แสดงทั้ง 8 ตัวอย่าง
# ------------------------------------------------------------

for i, (ref, pred) in enumerate(
    zip(refs, preds),
    start=1
):

    print(f"\nSample {i}")

    print("REF :", ref)
    print("PRED:", pred)

    print("-" * 70)

# หลัง Baseline Training

ถ้า:
- Sanity loss ลด
- Full train loss ลด
- validation loss ลดบางช่วง
- model เริ่ม output ภาษาไทยบางส่วน

ถือว่า **end-to-end lip reading pipeline ทำงานแล้ว**

ขั้นถัดไปที่แนะนำ:
1. เพิ่มวิดีโอ / ผู้พูด
2. ทำ transcript correction
3. ใช้ pretrained visual speech encoder
4. เปลี่ยน BiGRU → Transformer / Conformer
5. เพิ่ม data augmentation
6. ใช้ subword tokenizer แทน character-level
7. ทำ speaker-independent split
8. ประเมิน CER / WER อย่างจริงจัง